# IBKR Position Example

This example rebuilds a normalized position preview from a checked-in IBKR position artifact by default.

Set `SOURCE_MODE = "live"` only when TWS / IB Gateway is running locally and you want a fresh pull.

In [ ]:
from pathlib import Path
import sys
from typing import Dict

project_root = Path.cwd().resolve()
while not (project_root / "market_helper").exists():
    if project_root.parent == project_root:
        raise RuntimeError("Could not find project root containing the market_helper package.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


def load_local_account_defaults(root: Path) -> Dict[str, str]:
    candidates = [
        root / "configs" / "portfolio_monitor" / "local.env",
    ]
    values: Dict[str, str] = {}
    for path in candidates:
        if not path.exists():
            continue
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            values[key.strip()] = value.strip().strip(chr(34)).strip("'")
        break
    return values


def first_existing_path(candidates):
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]

local_account_defaults = load_local_account_defaults(project_root)
artifact_dir = project_root / "data" / "artifacts" / "portfolio_monitor"

SOURCE_MODE = "artifact_replay"  # "artifact_replay" or "live"

ARTIFACT_LIVE_CSV = first_existing_path([
    project_root / "outputs" / "reports" / "live_ibkr_position_report.csv",
    artifact_dir / "live_ibkr_position_report.csv",
])
POSITION_REPORT_OUTPUT = project_root / "outputs" / "reports" / "example_get_ibkr_position_preview.csv"

IB_HOST = "127.0.0.1"
IB_PORT = 7497
IB_CLIENT_ID = 123
IB_ACCOUNT = local_account_defaults.get("DEFAULT_PROD_ACCOUNT_ID", "")
SELECTED_SYMBOL = ""

print(f"Project root: {project_root}")
print(f"SOURCE_MODE: {SOURCE_MODE}")
print(f"Artifact CSV: {ARTIFACT_LIVE_CSV}")
print(f"Position report output: {POSITION_REPORT_OUTPUT}")
print(f"IBKR connection: host={IB_HOST}, port={IB_PORT}, client_id={IB_CLIENT_ID}, account={IB_ACCOUNT or '<default>'}")


In [ ]:
import csv

from IPython.display import display
import pandas as pd

from market_helper.portfolio.ibkr import normalize_ibkr_latest_prices, normalize_ibkr_positions
from market_helper.portfolio.security_reference import build_price_lookup, build_security_reference_table
from market_helper.presentation.tables.portfolio_report import build_position_report_rows


def rebuild_position_report_from_existing_live_csv(existing_csv_path: Path, output_path: Path):
    rows = list(csv.DictReader(existing_csv_path.open("r", encoding="utf-8", newline="")))
    if not rows:
        raise ValueError(f"No rows found in {existing_csv_path}")

    as_of = str(rows[0].get("as_of") or "") or None
    raw_positions = []
    raw_prices = []

    for row in rows:
        con_id = str(row.get("con_id") or "").strip()
        sec_type = "CASH" if str(row.get("symbol") or "").upper().endswith("_CASH_VALUE") else ("FUT" if str(row.get("exchange") or "").upper() in {"CBOT", "CFE", "CME", "COMEX", "ICE", "NYMEX"} else "STK")
        if not con_id and sec_type != "CASH":
            continue
        raw_positions.append({
            "account": str(row.get("account") or ""),
            "con_id": con_id,
            "sec_type": sec_type,
            "symbol": str(row.get("symbol") or ""),
            "currency": str(row.get("currency") or ""),
            "exchange": str(row.get("exchange") or ""),
            "local_symbol": str(row.get("local_symbol") or ""),
            "position": str(row.get("quantity") or "0"),
            "avg_cost": str(row.get("avg_cost") or ""),
            "market_value": str(row.get("market_value") or ""),
        })
        if con_id and str(row.get("latest_price") or "").strip():
            raw_prices.append({
                "con_id": con_id,
                "marketPrice": str(row.get("latest_price") or ""),
            })

    reference_table = build_security_reference_table()
    positions = normalize_ibkr_positions(raw_positions, reference_table, as_of=as_of)
    prices = normalize_ibkr_latest_prices(raw_prices, reference_table, as_of=as_of)
    report_rows = build_position_report_rows(positions, build_price_lookup(prices), reference_table.to_security_lookup())
    output_df = pd.DataFrame([row.__dict__ for row in report_rows])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_df.to_csv(output_path, index=False)
    return output_path, reference_table, output_df


if SOURCE_MODE == "live":
    from market_helper.workflows.generate_report import generate_live_ibkr_position_report

    position_report_path = generate_live_ibkr_position_report(
        output_path=POSITION_REPORT_OUTPUT,
        host=IB_HOST,
        port=IB_PORT,
        client_id=IB_CLIENT_ID,
        account_id=IB_ACCOUNT or None,
        timeout=4.0,
    )
    reference_table = build_security_reference_table()
    position_report_df = pd.read_csv(position_report_path)
    acquisition_note = "Generated directly from live TWS / IB Gateway."
else:
    position_report_path, reference_table, position_report_df = rebuild_position_report_from_existing_live_csv(
        existing_csv_path=ARTIFACT_LIVE_CSV,
        output_path=POSITION_REPORT_OUTPUT,
    )
    acquisition_note = "Rebuilt from an existing local live IBKR CSV artifact."

print(acquisition_note)
print(f"Position report path: {position_report_path}")
print(f"Position rows: {len(position_report_df)}")

preview_columns = [
    "account",
    "internal_id",
    "display_ticker",
    "display_name",
    "asset_class",
    "quantity",
    "market_value",
    "weight",
]
available_columns = [column for column in preview_columns if column in position_report_df.columns]
display(position_report_df.loc[:, available_columns].reset_index(drop=True))


In [ ]:
if position_report_df.empty or "display_ticker" not in position_report_df.columns:
    print("No normalized position rows were returned.")
    display(position_report_df)
else:
    if SELECTED_SYMBOL:
        selected_rows = position_report_df.loc[position_report_df["symbol"].eq(SELECTED_SYMBOL)]
        if selected_rows.empty:
            print(f"No row matched SELECTED_SYMBOL={SELECTED_SYMBOL}; showing the first row instead.")
            display(position_report_df.head(1).T)
        else:
            display(selected_rows.head(1).T)
    else:
        display(position_report_df.head(1).T)
